# 🏛️ Fine-Tuning de Modelos Soberanos con QLoRA y Unsloth
### Dominio: Auditoría de Cumplimiento Normativo (EU AI Act, DORA, ENS Alta, NIS2)

Este cuaderno permite entrenar un modelo **Llama 3 8B** o **Mistral 7B** utilizando **Unsloth** y **QLoRA a 4-bits**, reduciendo el consumo de VRAM en un 80% y duplicando la velocidad de entrenamiento en GPUs NVIDIA (T4, A10G, A100).

In [ ]:
# 1. Instalación de dependencias optimizadas
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# 2. Carga del Modelo Base con Unsloth (4-bit)
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None # Auto detección (Float16 o Bfloat16)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# 3. Configuración de Adaptadores LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 4. Carga y Formateo del Dataset Normativo
from datasets import load_dataset

dataset = load_dataset("json", data_files="dataset/train_chatml.json", split="train")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
# 5. Configuración del Trainer SFT
from transformers import TrainingArguments
from trl import SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

In [ ]:
# 6. Exportar a GGUF para Despliegue en Ollama / On-Premise
model.save_pretrained_gguf("sovereign-compliance-8b", tokenizer, quantization_method = "q4_k_m")